# 01 — LLM Behavior and Prompt Anatomy

## Scenario
Northstar’s classifier changes behavior after a request configuration change. This lab treats the request packet as an observable system and tests one variable at a time.

**Safety boundary:** this is a deterministic, credential-free simulator. It never calls a model, stores customer data, or executes an action. Its effects illustrate an experimental method, not a claim about a provider model.

## Learning objectives

- Identify the instruction, user data, evidence, examples, and decoding configuration in a request packet.
- Run a frozen case suite and interpret accuracy, support, and request-size results.
- Diagnose whether a change should be made in the prompt, context, configuration, or deterministic application boundary.

## Mental model

A model produces conditional next-token candidates from the visible request state. The application then validates the result. Prompts influence generation; they do not grant tool permission or replace validation.

```text
instruction + user data + context/examples + model/config → candidate output → validation → answer / clarification / escalation
```

## Setup and reproducibility

The reusable implementation is loaded directly from this course folder. Every experiment below uses the same four frozen cases. The `token_estimate` is a transparent word-count proxy for relative comparisons only; production code must use provider telemetry and tokenizer data.

In [1]:
from importlib.util import module_from_spec, spec_from_file_location
from pathlib import Path

module_path = Path.cwd() / 'curriculum/beginner/01-llm-behavior-and-prompt-anatomy/lab.py'
if not module_path.exists():
    module_path = Path.cwd() / 'lab.py'
spec = spec_from_file_location('prompt_anatomy_lab', module_path)
lab = module_from_spec(spec)
import sys
sys.modules[spec.name] = lab
spec.loader.exec_module(lab)
print({'cases': len(lab.CASES), 'module': str(module_path)})

{'cases': 4, 'module': '/Users/mahsateimourikia/Documents/Curated Repos/prompt-engineering/curriculum/beginner/01-llm-behavior-and-prompt-anatomy/lab.py'}


## Baseline hypothesis

A precise classification instruction, approved evidence, and no synthetic variation should classify clear cases while escalating the ambiguous payment case. We will score that baseline before changing anything.

In [2]:
def stable_packet(message):
    return lab.PromptPacket(
        instruction='Classify the support request using approved evidence.',
        user_message=message,
        evidence_position='first',
        evidence_available=True,
        temperature=0.0,
    )

baseline = lab.run_suite(stable_packet)
baseline_score = lab.score(baseline)
[(item.case_id, item.expected, item.observed) for item in baseline], baseline_score

([('clear-refund', 'refund', 'refund'),
  ('clear-shipping', 'shipping', 'shipping'),
  ('clear-account', 'account', 'account'),
  ('ambiguous-payment', 'unknown', 'unknown')],
 {'accuracy': 1.0, 'support_rate': 1.0, 'mean_token_estimate': 13.25})

## Inspect the baseline

The result is useful only because the expected label was defined independently. Notice that the ambiguous payment message remains `unknown`: forcing it into a confident category would inflate the wrong metric.

## Experiment 1 — position is a variable

Keep the cases and instruction fixed. Move the refund evidence into the synthetic middle position. This is a controlled proxy for a long-context position test; it tells us to test source order on a real model, not to assume a universal effect.

In [3]:
def middle_packet(message):
    return lab.PromptPacket(
        instruction='Classify the support request using approved evidence.',
        user_message=message,
        evidence_position='middle',
        evidence_available=True,
    )

middle = lab.run_suite(middle_packet)
middle_score = lab.score(middle)
[(item.case_id, item.expected, item.observed) for item in middle], middle_score

([('clear-refund', 'refund', 'unknown'),
  ('clear-shipping', 'shipping', 'shipping'),
  ('clear-account', 'account', 'account'),
  ('ambiguous-payment', 'unknown', 'unknown')],
 {'accuracy': 0.75, 'support_rate': 1.0, 'mean_token_estimate': 13.25})

## Experiment 2 — sampling is a trade-off

Now keep evidence first but introduce a non-zero synthetic temperature. On a real model, run repeated samples and compare variation, task accuracy, and the cost of additional calls. Never infer correctness from lower temperature alone.

In [4]:
def varied_packet(message):
    return lab.PromptPacket(
        instruction='Classify the support request using approved evidence.',
        user_message=message,
        evidence_position='first',
        evidence_available=True,
        temperature=0.9,
    )

varied = lab.run_suite(varied_packet)
varied_score = lab.score(varied)
[(item.case_id, item.expected, item.observed) for item in varied], varied_score

([('clear-refund', 'refund', 'refund'),
  ('clear-shipping', 'shipping', 'shipping'),
  ('clear-account', 'account', 'account'),
  ('ambiguous-payment', 'unknown', 'unknown')],
 {'accuracy': 1.0, 'support_rate': 1.0, 'mean_token_estimate': 13.25})

## Compare the experiment outcomes

This small table makes the decision inspectable. A production decision would add measured provider tokens, latency, cost, schema validity, and safety failures. Do not ship a setting because one example looks better.

In [5]:
comparison = {
    'baseline': baseline_score,
    'evidence_in_middle': middle_score,
    'higher_variation': varied_score,
}
comparison

{'baseline': {'accuracy': 1.0,
  'support_rate': 1.0,
  'mean_token_estimate': 13.25},
 'evidence_in_middle': {'accuracy': 0.75,
  'support_rate': 1.0,
  'mean_token_estimate': 13.25},
 'higher_variation': {'accuracy': 1.0,
  'support_rate': 1.0,
  'mean_token_estimate': 13.25}}

## Failure injection — missing evidence

A refund decision without approved evidence must not become a confident refund label. This is a context/contract failure, not a request for stronger role wording. The safe result is `unknown`, followed by clarification or escalation in the surrounding application.

In [6]:
missing_evidence = lab.PromptPacket(
    instruction='Classify the support request using approved evidence.',
    user_message='Can I return my order?',
    evidence_available=False,
)
outcome = lab.classify(missing_evidence)
assert outcome == 'unknown'
print({'observed': outcome, 'mitigation': 'clarify or escalate; do not infer eligibility'})

{'observed': 'unknown', 'mitigation': 'clarify or escalate; do not infer eligibility'}


## Production upgrade

| Teaching simulator | Production replacement |
| --- | --- |
| synthetic classifier | versioned provider adapter |
| word-count proxy | provider token/cost telemetry |
| fixed cases | versioned development, held-out, and regression suites |
| printed comparison | trace, dashboard, release gate, and rollback decision |

Record the model snapshot, decoding configuration, context identifiers, validation outcomes, latency, and cost. Keep identity, authorization, and effects outside the model.

## Exercises and challenge

1. Add an order-address message with an expected `account` label.
2. Add two examples to the packet and compare the request-size proxy with the baseline.
3. Design a real-model experiment that distinguishes a context-position regression from a model-version regression.
4. Challenge: define a release gate that prevents a lower-cost configuration from shipping if `unknown` correctness regresses.

## Summary
Prompt quality is measured behavior of an entire request packet. The next lesson turns that packet into an explicit instruction contract.